# Women's March Madness Pipeline
Same feature engineering as men's, but without Massey Ordinals or coach data.
Detailed data available from 2010+.

In [1]:
import sys
sys.path.insert(0, '..')

from src.data_loader import load_women_data
from src.feature_engineering import (
    compute_elo_ratings, elo_to_prob, parse_seeds,
    compute_season_stats, compute_conference_strength,
    build_team_features, build_matchup_matrix
)
from src.model import (
    leave_one_season_out_cv_lean, train_final_model,
    print_feature_importance, find_best_blend, LEAN_PARAMS
)

women = load_women_data('../data')
print('Loaded:', list(women.keys()))

Loaded: ['WComSsn', 'WDetSsn', 'WDetTrny', 'WTrnySeeds', 'WConf', 'WTeam']


## Feature Engineering

In [2]:
# Elo ratings
women_elo = compute_elo_ratings(women['WComSsn'])
print('Elo shape:', women_elo.shape)

# Seeds
women_seeds = parse_seeds(women['WTrnySeeds'])
print('Seeds shape:', women_seeds.shape)

# Season stats (detailed data from 2010+)
women_stats = compute_season_stats(women['WDetSsn'])
print('Stats shape:', women_stats.shape)
print('Stats nulls:', women_stats.isnull().sum().sum())

# Conference strength
women_conf = compute_conference_strength(women['WConf'], women_elo)
print('Conf shape:', women_conf.shape)

Elo shape: (9952, 3)
Seeds shape: (1744, 3)
Stats shape: (5965, 15)
Stats nulls: 0
Conf shape: (9853, 4)


In [3]:
# Build team features (no Massey for women)
women_features = build_team_features(
    women_elo,
    seeds_df=women_seeds,
    stats_df=women_stats,
    massey_df=None,
    conf_df=women_conf
)
print('Team features shape:', women_features.shape)
print('Columns:', women_features.columns.tolist())

Team features shape: (9952, 18)
Columns: ['Season', 'TeamID', 'Elo', 'SeedNum', 'eFG_off', 'TO_rate_off', 'OR_pct', 'FT_rate_off', 'eFG_def', 'TO_rate_def', 'DR_pct', 'FT_rate_def', 'Tempo', 'PPG', 'PPG_allowed', 'Win_pct', 'Last10_Win_pct', 'Conf_Elo_mean']


In [4]:
# Build matchup matrix from tournament games
women_matchups = build_matchup_matrix(women['WDetTrny'], women_features)
print('Matchups shape:', women_matchups.shape)
print('Target mean:', women_matchups['Target'].mean())
print('Nulls:', women_matchups.isnull().sum().sum())

Matchups shape: (961, 20)
Target mean: 0.5046826222684704
Nulls: 0


## Model Training & CV

In [5]:
# Women's lean features (no Massey columns)
WOMEN_LEAN_FEATURES = [
    'Elo_diff', 'SeedNum_diff',
    'eFG_off_diff', 'TO_rate_off_diff', 'OR_pct_diff', 'FT_rate_off_diff',
    'eFG_def_diff', 'TO_rate_def_diff', 'DR_pct_diff', 'FT_rate_def_diff',
    'Tempo_diff', 'PPG_diff', 'PPG_allowed_diff',
    'Win_pct_diff', 'Conf_Elo_mean_diff',
]

# Verify all features exist
available = [c for c in WOMEN_LEAN_FEATURES if c in women_matchups.columns]
print(f'Using {len(available)}/{len(WOMEN_LEAN_FEATURES)} features')

# Run CV
oof_women = leave_one_season_out_cv_lean(
    women_matchups,
    feature_cols=available
)

Using 15/15 features
Season 2010: Brier=0.17722  (n=63)
Season 2011: Brier=0.16734  (n=63)
Season 2012: Brier=0.12541  (n=63)
Season 2013: Brier=0.14160  (n=63)
Season 2014: Brier=0.13013  (n=63)
Season 2015: Brier=0.14061  (n=63)
Season 2016: Brier=0.17591  (n=63)
Season 2017: Brier=0.14558  (n=63)
Season 2018: Brier=0.14802  (n=63)
Season 2019: Brier=0.13798  (n=63)
Season 2021: Brier=0.15455  (n=63)
Season 2022: Brier=0.18370  (n=67)
Season 2023: Brier=0.18018  (n=67)
Season 2024: Brier=0.12577  (n=67)
Season 2025: Brier=0.11287  (n=67)

Overall OOF Brier (lean): 0.14980


In [6]:
# Blend with Elo
elo_a = oof_women.merge(women_elo, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'])['Elo']
elo_b = oof_women.merge(women_elo, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'])['Elo']
elo_probs = elo_to_prob(elo_a.values, elo_b.values)

best_w, best_brier = find_best_blend(oof_women, elo_probs)

w=0.00 (pure Elo):    Brier=0.15607
w=0.05:               Brier=0.15446
w=0.10:               Brier=0.15298
w=0.15:               Brier=0.15164
w=0.20:               Brier=0.15044
w=0.25:               Brier=0.14938
w=0.30:               Brier=0.14845
w=0.35:               Brier=0.14765
w=0.40:               Brier=0.14700
w=0.45:               Brier=0.14648
w=0.50:               Brier=0.14610
w=0.55:               Brier=0.14585
w=0.60:               Brier=0.14574
w=0.65:               Brier=0.14577
w=0.70:               Brier=0.14594
w=0.75:               Brier=0.14624
w=0.80:               Brier=0.14668
w=0.85:               Brier=0.14726
w=0.90:               Brier=0.14797
w=0.95:               Brier=0.14882
w=1.00:               Brier=0.14980 <-- pure XGB

Best blend: w=0.60, Brier=0.14574


In [7]:
# Train final model on all data
women_model, women_feat_cols = train_final_model(
    women_matchups,
    xgb_params=LEAN_PARAMS,
    num_boost_round=200
)
print_feature_importance(women_model)

Top features by importance:
  SeedNum_diff                   27.2484
  Elo_diff                       11.9984
  Conf_Elo_mean_diff             5.9875
  PPG_diff                       3.3460
  eFG_def_diff                   3.1020
  Win_pct_diff                   2.6359
  Last10_Win_pct_diff            2.6065
  PPG_allowed_diff               2.4405
  TO_rate_off_diff               2.4265
  OR_pct_diff                    2.4200
  eFG_off_diff                   2.3377
  FT_rate_def_diff               2.1319
  FT_rate_off_diff               2.1238
  DR_pct_diff                    1.9912
  TO_rate_def_diff               1.9839
